# Dataset D pancreas validation: scVelo dynamical RNA velocity

This notebook loads the official public CellRank pancreas preprocessed dataset, verifies the acquisition checksum recorded by notebook 00, runs scVelo dynamical fitting, computes velocity and velocity graph outputs, preserves projected velocity fields, and reports failed or weakly fitted genes.

No ScGeo methods, thresholds, frozen synthetic protocol settings, or simulation truth definitions are changed.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from pancreas_validation_common import (
    configured_paths,
    ensure_output_tree,
    ensure_runtime_env,
    load_config,
    rel_display,
    sha256_file,
    version_record,
    write_alt_text,
    write_dataframe,
    write_json,
    write_metadata,
)

CONFIG = load_config(ROOT)
PATHS = configured_paths(CONFIG, ROOT)
DATA_DIR = PATHS["data_dir"]
OUTPUT_DIR = PATHS["output_dir"]
DATA_DIR.mkdir(parents=True, exist_ok=True)
ensure_runtime_env(OUTPUT_DIR)
ensure_output_tree(OUTPUT_DIR)

In [ ]:

import json
from datetime import datetime, timezone

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

checksums = pd.read_csv(OUTPUT_DIR / "figure_sources" / "00_pancreas_dataset_checksums.csv")
pre_row = checksums.loc[checksums["kind"].eq(CONFIG["dataset"]["analysis_kind"])].iloc[0]
pre_path = ROOT / pre_row["path"]
if sha256_file(pre_path) != pre_row["sha256"]:
    raise AssertionError("Preprocessed pancreas checksum changed after acquisition")

adata = ad.read_h5ad(pre_path)
cluster_key = CONFIG["cluster_key"]
for layer in CONFIG["dataset"]["required_layers"]:
    if layer not in adata.layers:
        raise KeyError(f"Required layer missing: {layer}")
if cluster_key not in adata.obs:
    raise KeyError(f"Required annotation missing: {cluster_key}")

scv.settings.verbosity = 2
scv.settings.presenter_view = False
np.random.seed(CONFIG["scvelo"]["random_state"])

if "X_pca" not in adata.obsm or adata.obsm["X_pca"].shape[1] < CONFIG["scvelo"]["n_pcs"]:
    sc.tl.pca(adata, n_comps=CONFIG["scvelo"]["n_pcs"], svd_solver="arpack", random_state=CONFIG["scvelo"]["random_state"])
sc.pp.neighbors(
    adata,
    n_pcs=CONFIG["scvelo"]["n_pcs"],
    n_neighbors=CONFIG["scvelo"]["n_neighbors"],
    random_state=CONFIG["scvelo"]["random_state"],
)
scv.pp.moments(adata, n_pcs=CONFIG["scvelo"]["n_pcs"], n_neighbors=CONFIG["scvelo"]["n_neighbors"])

scv.tl.recover_dynamics(
    adata,
    max_iter=CONFIG["scvelo"]["recover_dynamics"]["max_iter"],
    n_jobs=CONFIG["scvelo"]["recover_dynamics"]["n_jobs"],
    show_progress_bar=True,
)
scv.tl.velocity(
    adata,
    mode=CONFIG["scvelo"]["velocity"]["mode"],
    min_r2=CONFIG["scvelo"]["velocity"]["min_r2"],
    min_likelihood=CONFIG["scvelo"]["velocity"]["min_likelihood"],
)
scv.tl.velocity_graph(
    adata,
    n_jobs=CONFIG["scvelo"]["velocity_graph"]["n_jobs"],
    show_progress_bar=True,
)
for basis in ["umap", "pca"]:
    if f"X_{basis}" in adata.obsm:
        scv.tl.velocity_embedding(adata, basis=basis, all_comps=True)

out_path = OUTPUT_DIR / "intermediates" / "pancreas_scvelo_dynamical.h5ad"
out_path.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(out_path, compression="gzip")

fit_cols = [col for col in adata.var.columns if col.startswith("fit_") or col in ["velocity_genes"]]
gene_fit = adata.var[fit_cols].copy() if fit_cols else pd.DataFrame(index=adata.var_names)
gene_fit.insert(0, "gene", adata.var_names)
for col in ["fit_likelihood", "fit_r2"]:
    if col not in gene_fit:
        gene_fit[col] = np.nan
if "velocity_genes" not in gene_fit:
    gene_fit["velocity_genes"] = False
weak_mask = (
    gene_fit["fit_likelihood"].astype(float).lt(CONFIG["scvelo"]["velocity"]["min_likelihood"]) |
    gene_fit["fit_r2"].astype(float).lt(CONFIG["scvelo"]["velocity"]["min_r2"])
)
failed_mask = gene_fit["fit_likelihood"].isna() | ~gene_fit["velocity_genes"].astype(bool)
gene_fit["fit_status"] = np.select(
    [failed_mask, weak_mask],
    ["failed_or_not_velocity_gene", "weak_fit_by_scvelo_default_cutoffs"],
    default="accepted_velocity_gene",
)

inventory_rows = []
for namespace, keys in [("layers", adata.layers.keys()), ("obsm", adata.obsm.keys()), ("obsp", adata.obsp.keys()), ("uns", adata.uns.keys())]:
    for key in sorted(keys):
        if "velocity" in key.lower() or key in ["Ms", "Mu", "moments"]:
            obj = getattr(adata, namespace)[key]
            inventory_rows.append({"namespace": namespace, "key": key, "shape": str(getattr(obj, "shape", "scalar"))})
inventory = pd.DataFrame(inventory_rows)
summary = pd.DataFrame([{
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_velocity_genes": int(gene_fit["velocity_genes"].astype(bool).sum()),
    "n_failed_or_not_velocity_gene": int(gene_fit["fit_status"].eq("failed_or_not_velocity_gene").sum()),
    "n_weak_fit": int(gene_fit["fit_status"].eq("weak_fit_by_scvelo_default_cutoffs").sum()),
    "output_h5ad": rel_display(out_path, ROOT),
    "output_sha256": sha256_file(out_path),
}])
write_dataframe(OUTPUT_DIR, "01_scvelo_gene_fit_status", gene_fit)
write_dataframe(OUTPUT_DIR, "01_velocity_output_inventory", inventory)
write_dataframe(OUTPUT_DIR, "01_scvelo_summary", summary)
write_alt_text(
    OUTPUT_DIR,
    "01_scvelo_dynamical_velocity",
    f"scVelo dynamical velocity was computed for the public pancreas preprocessed dataset. {int(gene_fit['fit_status'].eq('failed_or_not_velocity_gene').sum())} genes were failed or not selected as velocity genes and {int(gene_fit['fit_status'].eq('weak_fit_by_scvelo_default_cutoffs').sum())} genes were weak by scVelo default likelihood or R-squared cutoffs."
)
write_metadata(OUTPUT_DIR, "01_scvelo_dynamical_velocity", CONFIG, {
    "input_h5ad": rel_display(pre_path, ROOT),
    "input_sha256": pre_row["sha256"],
    "output_h5ad": rel_display(out_path, ROOT),
    "output_sha256": sha256_file(out_path),
    "scvelo_parameters": CONFIG["scvelo"],
})
version_record(OUTPUT_DIR, "01_scvelo_dynamical_velocity", CONFIG, {"output_h5ad_sha256": sha256_file(out_path)})
summary
